# API Fundamentals

**Module:** 08-llm-apis

**Notebook:** `01-api-fundamentals.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **What is an LLM API?** with clear contracts and failure modes
- Explain and apply **Credentials & Safety** with clear contracts and failure modes
- Explain and apply **Messages vs Prompts** with clear contracts and failure modes
- Explain and apply **Response Shape** with clear contracts and failure modes
- Explain and apply **Request Pattern** with clear contracts and failure modes
- Explain and apply **Errors** with clear contracts and failure modes
- Explain and apply **App Mental Model** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — API Fundamentals

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **What is an LLM API?**
2. **Credentials & Safety**
3. **Messages vs Prompts**
4. **Response Shape**
5. **Request Pattern**
6. **Errors**
7. **App Mental Model**

Read top-to-bottom once, then revisit weak spots with the exercises.


## What is an LLM API?

### Definition
**What is an LLM API?** is a core building block in 01-api-fundamentals within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around What is an LLM API? typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For What is an LLM API?: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain What is an LLM API? as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating What is an LLM API? as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for What is an LLM API?
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use What is an LLM API? when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does What is an LLM API? improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "What is an LLM API?" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "What is an LLM API?"
    notebook: str = "01-api-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
import os

def build_chat_request(model: str, user: str, system: str = "You are concise."):
    return {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0.2,
    }

headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}", "Content-Type": "application/json"}
fake_response = {
    "id": "chatcmpl_demo",
    "choices": [{"message": {"role": "assistant", "content": "OK"}, "finish_reason": "stop"}],
    "usage": {"prompt_tokens": 42, "completion_tokens": 1, "total_tokens": 43},
}
print(build_chat_request("gpt-4.1-mini", "ping")["model"])
print("auth:", headers["Authorization"][:20] + "...", "usage:", fake_response["usage"])


In [ ]:
# Multi-provider router stub
PROVIDERS = {
    "openai": {"base": "https://api.openai.com/v1", "env": "OPENAI_API_KEY"},
    "anthropic": {"base": "https://api.anthropic.com/v1", "env": "ANTHROPIC_API_KEY"},
    "gemini": {"base": "https://generativelanguage.googleapis.com", "env": "GOOGLE_API_KEY"},
}

def resolve_provider(name: str) -> dict:
    p = PROVIDERS[name]
    import os
    return {"base": p["base"], "api_key": os.environ.get(p["env"], "YOUR_API_KEY")}

print({k: resolve_provider(k)["api_key"][:12] + "..." for k in PROVIDERS})


## Credentials & Safety

### Definition
**Credentials & Safety** is a core building block in 01-api-fundamentals within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Credentials & Safety typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Credentials & Safety: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Credentials & Safety as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Credentials & Safety as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Credentials & Safety
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Credentials & Safety when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Credentials & Safety" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Credentials & Safety"
    notebook: str = "01-api-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
import os

def build_chat_request(model: str, user: str, system: str = "You are concise."):
    return {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0.2,
    }

headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}", "Content-Type": "application/json"}
fake_response = {
    "id": "chatcmpl_demo",
    "choices": [{"message": {"role": "assistant", "content": "OK"}, "finish_reason": "stop"}],
    "usage": {"prompt_tokens": 42, "completion_tokens": 1, "total_tokens": 43},
}
print(build_chat_request("gpt-4.1-mini", "ping")["model"])
print("auth:", headers["Authorization"][:20] + "...", "usage:", fake_response["usage"])


In [ ]:
# Multi-provider router stub
PROVIDERS = {
    "openai": {"base": "https://api.openai.com/v1", "env": "OPENAI_API_KEY"},
    "anthropic": {"base": "https://api.anthropic.com/v1", "env": "ANTHROPIC_API_KEY"},
    "gemini": {"base": "https://generativelanguage.googleapis.com", "env": "GOOGLE_API_KEY"},
}

def resolve_provider(name: str) -> dict:
    p = PROVIDERS[name]
    import os
    return {"base": p["base"], "api_key": os.environ.get(p["env"], "YOUR_API_KEY")}

print({k: resolve_provider(k)["api_key"][:12] + "..." for k in PROVIDERS})


### Worked scenario — Credentials & Safety

**Situation:** A team wants to productionize a feature involving **Credentials & Safety**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Messages vs Prompts

### Definition
**Messages vs Prompts** helps you choose among alternatives using explicit criteria rather than hype.

### Why it matters
In LLM API integration, weak designs around Messages vs Prompts typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
List options, define criteria (quality, cost, latency, ops, lock-in), score with evidence, document the decision.

### Intuition
Explain Messages vs Prompts as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Messages vs Prompts as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Messages vs Prompts
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Messages vs Prompts when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Messages vs Prompts" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Messages vs Prompts"
    notebook: str = "01-api-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# MCP-like component registry (pedagogical)
registry = {
    "resources": [{"uri": "doc://readme", "name": "README", "mimeType": "text/plain"}],
    "tools": [{"name": "search", "inputSchema": {"type": "object", "properties": {"q": {"type": "string"}}}}],
    "prompts": [{"name": "explain", "arguments": [{"name": "topic"}]}],
}

def list_caps():
    return {k: [x.get("name") or x.get("uri") for x in v] for k, v in registry.items()}

print(list_caps())


In [ ]:
# JSON-RPC style message shapes used conceptually by MCP
msg_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search", "arguments": {"q": "SSO"}}}
msg_response = {"jsonrpc": "2.0", "id": 1, "result": {"content": [{"type": "text", "text": "SSO allowlist..."}]}}
msg_error = {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}
print(msg_request["method"], "=>", msg_response["result"]["content"][0]["text"][:40])


## Response Shape

### Definition
**Response Shape** is a core building block in 01-api-fundamentals within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Response Shape typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Response Shape: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Response Shape as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Response Shape as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Response Shape
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Response Shape when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Response Shape" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Response Shape"
    notebook: str = "01-api-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Response Shape"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Response Shape"}
strong = {"definition": "Response Shape", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Response Shape"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Response Shape", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Response Shape

**Situation:** A team wants to productionize a feature involving **Response Shape**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Request Pattern

### Definition
**Request Pattern** is a core building block in 01-api-fundamentals within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Request Pattern typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Request Pattern: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Request Pattern as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Request Pattern as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Request Pattern
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Request Pattern when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Request Pattern" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Request Pattern"
    notebook: str = "01-api-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Request Pattern"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Request Pattern"}
strong = {"definition": "Request Pattern", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Request Pattern"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Request Pattern", "passed": len(checks)-len(failed), "failed": failed})


In [ ]:
# Demo: decision table for applying "Request Pattern"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_request_patt", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


## Errors

### Definition
**Errors** is a core building block in 01-api-fundamentals within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Errors typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Errors: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Errors as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Errors as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Errors
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Errors when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Errors" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Errors"
    notebook: str = "01-api-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Errors"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Errors"}
strong = {"definition": "Errors", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Errors"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Errors", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Errors

**Situation:** A team wants to productionize a feature involving **Errors**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## App Mental Model

### Definition
**App Mental Model** is a core building block in 01-api-fundamentals within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around App Mental Model typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For App Mental Model: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain App Mental Model as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating App Mental Model as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for App Mental Model
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use App Mental Model when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "App Mental Model" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "App Mental Model"
    notebook: str = "01-api-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "App Mental Model"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "App Mental Model"}
strong = {"definition": "App Mental Model", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "App Mental Model"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "App Mental Model", "passed": len(checks)-len(failed), "failed": failed})


## Comparison Snapshot

Use this table when reviewing designs in **API Fundamentals**.

| Topic | Do | Don't |
|-------|----|-------|
| What is an LLM API? | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Credentials & Safety | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Messages vs Prompts | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Response Shape | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Request Pattern | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Errors | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| What is an LLM API? | Key concept covered in this notebook; see its section for definition and pitfalls |
| Credentials & Safety | Key concept covered in this notebook; see its section for definition and pitfalls |
| Messages vs Prompts | Key concept covered in this notebook; see its section for definition and pitfalls |
| Response Shape | Key concept covered in this notebook; see its section for definition and pitfalls |
| Request Pattern | Key concept covered in this notebook; see its section for definition and pitfalls |
| Errors | Key concept covered in this notebook; see its section for definition and pitfalls |
| App Mental Model | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **API Fundamentals** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **08-llm-apis**.


## Try It Yourself

1. Implement a failing test/fixture for **What is an LLM API?**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Credentials & Safety**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Messages vs Prompts**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Response Shape**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Request Pattern**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
